In [1]:
import duckdb
import pandas as pd
from pathlib import Path

DATA_DIR = Path("..") / "data"
CLEAN = (DATA_DIR / "clean_sales.parquet").as_posix()

print(Path.cwd())

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE TABLE clean_sales AS SELECT * FROM read_parquet('{CLEAN}')")
print(con.execute("SELECT COUNT(*) FROM clean_sales").fetchall()[0][0], "rows loaded")

C:\Users\kanak\retail-customer-analytics\notebooks
1021272 rows loaded


In [2]:
VOID_INVOICES = ["541431", "C541433", "581483", "C581484"]
voids = ",".join(f"'{i}'" for i in VOID_INVOICES)

con.execute(f"""
CREATE OR REPLACE TABLE purchases AS
SELECT DISTINCT
    customer_id,
    CAST(DATE_TRUNC('month', invoice_ts) AS DATE) AS activity_month
FROM clean_sales
WHERE customer_id IS NOT NULL
  AND NOT is_return
  AND invoice NOT IN ({voids})
  AND invoice_ts < TIMESTAMP '2011-12-01'
""")
print(con.execute("SELECT COUNT(*) FROM purchases").fetchall()[0][0], "customer-months")

24888 customer-months


In [3]:
con.execute("""
CREATE OR REPLACE TABLE cohorts AS
SELECT customer_id, MIN(activity_month) AS cohort_month
FROM purchases
GROUP BY customer_id
""")

print(con.execute("SELECT COUNT(*) FROM cohorts").fetchall()[0][0], "customers in cohorts")

only_dec_2011 = con.execute(f"""
SELECT COUNT(*) FROM (
    SELECT customer_id FROM clean_sales
    WHERE customer_id IS NOT NULL AND NOT is_return AND invoice NOT IN ({voids})
    GROUP BY customer_id
    HAVING MIN(invoice_ts) >= TIMESTAMP '2011-12-01'
)""").fetchall()[0][0]
print(only_dec_2011, "customers whose first purchase is in Dec 2011 (excluded)")

5824 customers in cohorts
28 customers whose first purchase is in Dec 2011 (excluded)


In [4]:
con.execute("""
CREATE OR REPLACE TABLE cohort_activity AS
SELECT c.cohort_month,
       DATE_DIFF('month', c.cohort_month, p.activity_month) AS month_index,
       COUNT(DISTINCT p.customer_id) AS active_customers
FROM purchases p
JOIN cohorts c ON c.customer_id = p.customer_id
GROUP BY 1, 2
""")

con.execute("""
CREATE OR REPLACE TABLE cohort_retention AS
WITH sizes AS (
    SELECT cohort_month, COUNT(*) AS cohort_size FROM cohorts GROUP BY 1
),
grid AS (
    SELECT s.cohort_month, s.cohort_size, t.k AS month_index
    FROM sizes s, generate_series(0, 23) AS t(k)
    WHERE t.k <= DATE_DIFF('month', s.cohort_month, DATE '2011-11-01')
)
SELECT g.cohort_month, g.month_index, g.cohort_size,
       COALESCE(a.active_customers, 0) AS active_customers,
       ROUND(100.0 * COALESCE(a.active_customers, 0) / g.cohort_size, 1) AS retention_pct
FROM grid g
LEFT JOIN cohort_activity a
  ON a.cohort_month = g.cohort_month AND a.month_index = g.month_index
""")

print(con.execute("SELECT COUNT(*) FROM cohort_retention").fetchall()[0][0], "cohort-month cells")

300 cohort-month cells


In [5]:
ret = con.execute("SELECT * FROM cohort_retention ORDER BY cohort_month, month_index").fetchdf()
ret["cohort_month"] = ret["cohort_month"].astype(str).str[:7]

pivot = ret.pivot(index="cohort_month", columns="month_index", values="retention_pct")
sizes = ret.groupby("cohort_month")["cohort_size"].first().rename("size")

display(pd.concat([sizes, pivot[[0, 1, 2, 3, 6, 12]]], axis=1))

,size,0,1,2,3,6,12
cohort_month,,,,,,,
2009-12,951,100.0,35.0,33.3,42.5,37.6,37.6
2010-01,368,100.0,21.5,32.1,31.5,26.9,22.8
2010-02,375,100.0,23.5,22.7,29.3,19.2,15.2
2010-03,441,100.0,19.0,23.1,24.3,24.7,20.0
2010-04,294,100.0,19.0,19.0,16.0,27.6,14.3
2010-05,255,100.0,15.7,16.9,17.6,21.2,15.3
2010-06,267,100.0,17.6,18.7,20.6,12.7,15.0
2010-07,185,100.0,15.7,18.4,29.7,11.4,13.0
2010-08,163,100.0,19.6,28.8,32.5,9.8,14.7


In [6]:
con.sql("""
SELECT month_index,
       COUNT(*)                AS cohorts_included,
       SUM(cohort_size)        AS customers,
       ROUND(100.0 * SUM(active_customers) / SUM(cohort_size), 1) AS pooled_retention_pct
FROM cohort_retention
WHERE cohort_month > DATE '2009-12-01'
  AND month_index IN (0, 1, 2, 3, 6, 9, 12)
GROUP BY 1
ORDER BY 1
""").show()

┌─────────────┬──────────────────┬───────────┬──────────────────────┐
│ month_index │ cohorts_included │ customers │ pooled_retention_pct │
│    int64    │      int64       │  int128   │        double        │
├─────────────┼──────────────────┼───────────┼──────────────────────┤
│           0 │               23 │      4873 │                100.0 │
│           1 │               22 │      4682 │                 21.0 │
│           2 │               21 │      4461 │                 21.5 │
│           3 │               20 │      4273 │                 21.1 │
│           6 │               17 │      3957 │                 18.4 │
│           9 │               14 │      3561 │                 15.0 │
│          12 │               11 │      3288 │                 18.4 │
└─────────────┴──────────────────┴───────────┴──────────────────────┘



In [7]:
con.sql("""
SELECT month_index,
       SUM(cohort_size)       AS customers,
       SUM(active_customers)  AS active,
       ROUND(100.0 * SUM(active_customers) / SUM(cohort_size), 1) AS retention_pct
FROM cohort_retention
WHERE cohort_month BETWEEN DATE '2010-01-01' AND DATE '2010-11-01'
  AND month_index <= 12
GROUP BY 1
ORDER BY 1
""").show()

┌─────────────┬───────────┬────────┬───────────────┐
│ month_index │ customers │ active │ retention_pct │
│    int64    │  int128   │ int128 │    double     │
├─────────────┼───────────┼────────┼───────────────┤
│           0 │      3288 │   3288 │         100.0 │
│           1 │      3288 │    664 │          20.2 │
│           2 │      3288 │    677 │          20.6 │
│           3 │      3288 │    697 │          21.2 │
│           4 │      3288 │    615 │          18.7 │
│           5 │      3288 │    614 │          18.7 │
│           6 │      3288 │    610 │          18.6 │
│           7 │      3288 │    619 │          18.8 │
│           8 │      3288 │    539 │          16.4 │
│           9 │      3288 │    493 │          15.0 │
│          10 │      3288 │    437 │          13.3 │
│          11 │      3288 │    455 │          13.8 │
│          12 │      3288 │    606 │          18.4 │
└─────────────┴───────────┴────────┴───────────────┘
  13 rows                                4 col

In [8]:
con.sql("""
WITH first_return AS (
    SELECT c.customer_id,
           MIN(DATE_DIFF('month', c.cohort_month, p.activity_month)) AS first_return_idx
    FROM cohorts c
    JOIN purchases p
      ON p.customer_id = c.customer_id AND p.activity_month > c.cohort_month
    GROUP BY c.customer_id
),
base AS (
    SELECT c.customer_id, f.first_return_idx
    FROM cohorts c
    LEFT JOIN first_return f ON f.customer_id = c.customer_id
    WHERE c.cohort_month BETWEEN DATE '2010-01-01' AND DATE '2010-11-01'
)
SELECT t.k AS months_after_first_purchase,
       COUNT(*) AS customers,
       COUNT(*) FILTER (WHERE first_return_idx <= t.k) AS returned_by_k,
       ROUND(100.0 * COUNT(*) FILTER (WHERE first_return_idx <= t.k) / COUNT(*), 1) AS returned_pct
FROM base, (VALUES (1), (3), (6), (12)) AS t(k)
GROUP BY t.k
ORDER BY t.k
""").show()

┌─────────────────────────────┬───────────┬───────────────┬──────────────┐
│ months_after_first_purchase │ customers │ returned_by_k │ returned_pct │
│            int32            │   int64   │     int64     │    double    │
├─────────────────────────────┼───────────┼───────────────┼──────────────┤
│                           1 │      3288 │           664 │         20.2 │
│                           3 │      3288 │          1385 │         42.1 │
│                           6 │      3288 │          1887 │         57.4 │
│                          12 │      3288 │          2301 │         70.0 │
└─────────────────────────────┴───────────┴───────────────┴──────────────┘



In [9]:
OUT = (DATA_DIR / "cohort_retention.csv").as_posix()
con.execute(f"COPY cohort_retention TO '{OUT}' (FORMAT CSV, HEADER)")
print("saved", OUT)

saved ../data/cohort_retention.csv
